# DSPy + TruLens: Optimize a RAG Program with TruLens Metrics

[DSPy](https://dspy.ai) optimizers improve a program's instructions and
few-shot demos against a *metric*. TruLens metrics such as groundedness and
answer relevance already score RAG answers on a [0, 1] scale without reference
answers, which makes them natural optimization targets.

This notebook shows how to:

1. Build a small DSPy RAG program over an in-memory corpus.
2. Wrap TruLens groundedness and answer relevance as DSPy metrics with
   `TruDSPyMetric`, mapping each metric argument to a DSPy field explicitly.
3. Evaluate the baseline program with `dspy.Evaluate` and optimize it with
   `dspy.MIPROv2`.
4. Record the baseline and optimized programs as two versions of one TruLens
   app and compare them in the leaderboard.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/truera/trulens/blob/main/examples/expositional/frameworks/dspy/dspy_trulens_metrics.ipynb)

In [ ]:
# MIPROv2 needs DSPy's optuna extra.
# !pip install trulens trulens-apps-dspy trulens-providers-openai "dspy[optuna]"

## Setup

In [ ]:
import os

# Set your OpenAI API key.
os.environ["OPENAI_API_KEY"] = "sk-..."  # replace with your key

## 1. Configure DSPy

In [ ]:
import dspy

dspy.configure(lm=dspy.LM("openai/gpt-4o-mini"))

## 2. Build a small RAG program

The corpus describes a fictional bike-share service, so the model cannot answer
from memory and groundedness matters. A keyword-overlap retriever keeps the
example free of vector stores.

`@instrument` marks `search` as a retrieval span so TruLens can read the
retrieved passages when the programs are recorded in step 7. Outside a TruLens
recording it does nothing.

In [ ]:
import re

from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes

CORPUS = [
    "Riverside Bikes is the city's bike-share service. It runs 120 docking "
    "stations and about 1,500 bikes, a third of them electric.",
    "A Riverside Bikes day pass costs $8 and includes unlimited 45-minute "
    "rides for 24 hours. Longer rides cost $2 for each extra 15 minutes.",
    "An annual Riverside Bikes membership costs $90 and includes unlimited "
    "60-minute rides. Members pay $3 for each extra 15 minutes.",
    "Electric bikes cost an extra $0.15 per minute for every rider, including "
    "members and day-pass holders.",
    "Riverside Bikes stations are open 24 hours a day, every day. Electric "
    "bikes are taken out of service when it is colder than -10 degrees "
    "Celsius.",
    "Riders must be at least 16 years old. Riverside Bikes does not provide a "
    "helmet, but city law requires riders under 18 to wear one.",
    "If a dock is full, riders can tap the station kiosk to get 15 extra "
    "minutes at no charge to reach another station.",
    "A bike that is not returned within 24 hours is reported lost, and the "
    "rider is charged a $1,200 replacement fee.",
]


def tokenize(text: str) -> set[str]:
    """Lowercase words of three or more characters."""
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2}


@instrument(
    span_type=SpanAttributes.SpanType.RETRIEVAL,
    attributes={
        SpanAttributes.RETRIEVAL.QUERY_TEXT: "query",
        SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
    },
)
def search(query: str, k: int = 2) -> list[str]:
    """Return the k passages that share the most words with the query."""
    words = tokenize(query)
    return sorted(CORPUS, key=lambda p: -len(words & tokenize(p)))[:k]

In [ ]:
class RAG(dspy.Module):
    def __init__(self, k: int = 2):
        super().__init__()
        self.k = k
        self.respond = dspy.ChainOfThought("context, question -> answer")

    def forward(self, question: str) -> dspy.Prediction:
        context = search(question, k=self.k)
        answer = self.respond(context=context, question=question).answer
        # Return the passages too, so metrics can check the answer against them.
        return dspy.Prediction(context=context, answer=answer)


baseline = RAG()
baseline(question="How much does a day pass cost?")

## 3. Create train and dev sets

Both TruLens metrics are reference-free, so each example needs only a question.
`with_inputs("question")` tells DSPy which fields to pass to the program.
MIPROv2 holds out part of the trainset for its own validation, so the devset
stays unseen until the final comparison.

In [ ]:
train_questions = [
    "How many docking stations does Riverside Bikes have?",
    "What does an annual membership include?",
    "How much extra do electric bikes cost?",
    "Are the stations open at night?",
    "How old do you have to be to ride?",
    "What happens if I never return a bike?",
]
dev_questions = [
    "How long can I ride on a day pass before paying extra?",
    "Do members pay extra for electric bikes?",
    "What should I do if the dock is full?",
    "Do I need to bring my own helmet?",
]

trainset = [
    dspy.Example(question=q).with_inputs("question") for q in train_questions
]
devset = [
    dspy.Example(question=q).with_inputs("question") for q in dev_questions
]

## 4. Wrap TruLens metrics as DSPy metrics

`TruDSPyMetric` puts a TruLens `Metric` behind the DSPy metric interface.
`fields` maps each argument of the TruLens implementation to the DSPy value it
reads, and nothing is inferred from argument names:

- `groundedness_measure_with_cot_reasons(source, statement)` checks the answer
  against the passages the program retrieved. They are a list on the
  prediction, so a callable joins them into one `source`.
- `relevance_with_cot_reasons(prompt, response)` checks the answer against the
  question on the example.

In [ ]:
from trulens.apps.dspy import TruDSPyMetric
from trulens.core import Metric
from trulens.providers.openai import OpenAI

provider = OpenAI(model_engine="gpt-4o-mini")

groundedness_metric = Metric(
    implementation=provider.groundedness_measure_with_cot_reasons,
    name="Groundedness",
)
answer_relevance_metric = Metric(
    implementation=provider.relevance_with_cot_reasons,
    name="Answer Relevance",
)

groundedness = TruDSPyMetric(
    groundedness_metric,
    fields={
        "source": lambda example, prediction: "\n\n".join(prediction.context),
        "statement": "prediction.answer",
    },
)
answer_relevance = TruDSPyMetric(
    answer_relevance_metric,
    fields={"prompt": "example.question", "response": "prediction.answer"},
)

In [ ]:
example = devset[0]
prediction = baseline(**example.inputs())

print("Answer:          ", prediction.answer)
print("Groundedness:    ", groundedness(example, prediction))
print("Answer relevance:", answer_relevance(example, prediction))

DSPy optimizes one number, so average the two scores into a single metric.

In [ ]:
def rag_quality(example, prediction, trace=None):
    """Mean of groundedness and answer relevance, in [0, 1]."""
    return (
        groundedness(example, prediction)
        + answer_relevance(example, prediction)
    ) / 2

## 5. Evaluate the baseline

`dspy.Evaluate` reports the mean metric value as a percentage.

In [ ]:
evaluate = dspy.Evaluate(
    devset=devset, metric=rag_quality, display_progress=True, display_table=True
)

baseline_result = evaluate(baseline)

## 6. Optimize with MIPROv2

`auto="light"` keeps the search small. The examples carry no reference answers,
so `max_labeled_demos=0`, and MIPROv2 bootstraps its few-shot demos from the
program's own runs.

In [ ]:
optimizer = dspy.MIPROv2(metric=rag_quality, auto="light")
optimized = optimizer.compile(
    RAG(), trainset=trainset, max_bootstrapped_demos=2, max_labeled_demos=0
)

In [ ]:
optimized_result = evaluate(optimized)

print(f"Baseline:  {baseline_result.score:.1f}")
print(f"Optimized: {optimized_result.score:.1f}")

In [ ]:
for name, predictor in optimized.named_predictors():
    print(f"{name} ({len(predictor.demos)} demos)")
    print(predictor.signature.instructions)

### Other optimizers

The same metrics work with DSPy's other optimizers.

Bootstrap optimizers such as `dspy.BootstrapFewShot` call the metric with a
`trace` and keep a demo when the result is truthy. Set `threshold` to return
`score >= threshold` as a bool on those calls:

```python
strict_groundedness = TruDSPyMetric(
    groundedness_metric,
    fields={
        "source": lambda example, prediction: "\n\n".join(prediction.context),
        "statement": "prediction.answer",
    },
    threshold=0.8,
)

bootstrap = dspy.BootstrapFewShot(
    metric=strict_groundedness, max_labeled_demos=0
)
bootstrapped = bootstrap.compile(RAG(), trainset=trainset)
```

`dspy.GEPA` also asks the metric for feedback on each predictor. On those calls
a `TruDSPyMetric` returns the TruLens reasons alongside the score, which GEPA's
reflection model reads when it proposes new instructions:

```python
gepa = dspy.GEPA(
    metric=groundedness,
    auto="light",
    reflection_lm=dspy.LM("openai/gpt-5", temperature=1.0, max_tokens=32000),
)
evolved = gepa.compile(RAG(), trainset=trainset)
```

## 7. Compare the baseline and optimized programs in TruLens

`TruDSPyMetric` scores predictions while DSPy optimizes, but it does not record
optimizer candidates. To keep a record of the outcome, run the baseline and
optimized programs on the devset as two versions of the same TruLens app.

The same two `Metric` objects score the TruLens records. Only the argument
wiring changes: selectors read the retrieved passages, the question and the
answer from the recorded spans instead of from DSPy fields.

In [ ]:
from trulens.core import Selector
from trulens.core import TruSession

session = TruSession()
session.reset_database()

f_groundedness = groundedness_metric.on({
    "source": Selector.select_context(collect_list=True),
    "statement": Selector.select_record_output(),
})
f_answer_relevance = answer_relevance_metric.on({
    "prompt": Selector.select_record_input(),
    "response": Selector.select_record_output(),
})

`RecordedRAG` gives TruLens a string-in, string-out entry point around a DSPy
program, so the record input is the question and the record output is the
answer.

In [ ]:
from trulens.apps.app import TruApp


class RecordedRAG:
    def __init__(self, program: dspy.Module):
        self.program = program

    @instrument(
        span_type=SpanAttributes.SpanType.RECORD_ROOT,
        attributes={
            SpanAttributes.RECORD_ROOT.INPUT: "question",
            SpanAttributes.RECORD_ROOT.OUTPUT: "return",
        },
    )
    def answer(self, question: str) -> str:
        return self.program(question=question).answer


for version, program in [("baseline", baseline), ("mipro", optimized)]:
    rag_app = RecordedRAG(program)
    tru_rag = TruApp(
        rag_app,
        app_name="DSPy RAG",
        app_version=version,
        feedbacks=[f_groundedness, f_answer_relevance],
    )
    with tru_rag as recording:
        for example in devset:
            rag_app.answer(example.question)
    recording.retrieve_feedback_results()

## 8. Check results

The leaderboard shows the mean groundedness and answer relevance for each app
version.

In [ ]:
session.get_leaderboard()

In [ ]:
from trulens.dashboard import run_dashboard

run_dashboard(session)